In [1]:
import os

VSL400 = (os.listdir('/kaggle/input/datasets/bowboochua9/vsl400'))

# VSL = (os.listdir('/kaggle/input/datasets/aresusayhi/vsl-vietnamese-sign-languages'))

In [2]:
!mkdir -p /kaggle/working/keypoints

!cp -r /kaggle/input/datasets/ngcsnchu/endtomid/kaggle/working/keypoints/* \
/kaggle/working/keypoints/

In [3]:
!rm -rf /kaggle/working/keypoints
!mkdir -p /kaggle/working/keypoints/train
!mkdir -p /kaggle/working/keypoints/test

In [4]:
import os

npy_files = []

for root, dirs, files in os.walk("/kaggle/working/keypoints"):
    for f in files:
        if f.endswith(".npy"):
            npy_files.append(f)

npy_files = sorted(npy_files)

for f in npy_files:
    print(f)

print("Số npy trong working:", len(npy_files))

Số npy trong working: 0


In [5]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 10.8 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.6 which is incompatible.


In [6]:
!mkdir -p /kaggle/working/mp_models

!wget -q -O /kaggle/working/mp_models/pose_landmarker.task \
https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task

!wget -q -O /kaggle/working/mp_models/hand_landmarker.task \
https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task

!ls -lh /kaggle/working/mp_models

total 13M
-rw-r--r-- 1 root root 7.5M Apr 26  2023 hand_landmarker.task
-rw-r--r-- 1 root root 5.6M Apr 27  2023 pose_landmarker.task


In [7]:
import os
import json
import pandas as pd

BASE_DIR = "/kaggle/input/datasets/bowboochua9/vsl400"

# Nếu train webcam chính diện thì dùng front_view trước
VIEWS = ["front_view"]

# Nếu muốn lấy cả 3 view thì đổi thành:
# VIEWS = ["front_view", "left_view", "right_view"]

TRAIN_SPLITS = [1, 2, 3, 4, 5, 6]
TEST_SPLITS = [7]


def load_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)


def normalize_video_name(video_id):
    video_id = str(video_id)

    if video_id.endswith(".mp4"):
        return video_id

    if video_id.isdigit():
        return video_id.zfill(6) + ".mp4"

    return video_id + ".mp4"


def extract_rows_from_json(data):
    rows = []

    if isinstance(data, list):
        for item in data:
            if not isinstance(item, dict):
                continue

            video_id = (
                item.get("video_id")
                or item.get("id")
                or item.get("name")
                or item.get("file")
                or item.get("filename")
            )

            gloss = (
                item.get("gloss")
                or item.get("label")
                or item.get("text")
                or item.get("word")
                or item.get("class")
            )

            rows.append({
                "video_id": video_id,
                "gloss": gloss
            })

    elif isinstance(data, dict):
        for key, value in data.items():

            if isinstance(value, str):
                rows.append({
                    "video_id": key,
                    "gloss": value
                })

            elif isinstance(value, dict):
                video_id = (
                    value.get("video_id")
                    or value.get("id")
                    or value.get("name")
                    or value.get("file")
                    or value.get("filename")
                    or key
                )

                gloss = (
                    value.get("gloss")
                    or value.get("label")
                    or value.get("text")
                    or value.get("word")
                    or value.get("class")
                )

                rows.append({
                    "video_id": video_id,
                    "gloss": gloss
                })

    return rows


def get_split_dir(split_num):
    """
    Dataset dạng:
    Part_1/split_1
    Part_2/split_2
    ...
    Part_7/split_7
    """
    return os.path.join(BASE_DIR, f"Part_{split_num}", f"split_{split_num}")


def build_metadata(split_list, split_type):
    all_rows = []

    for split_num in split_list:
        split_name = f"split_{split_num}"
        part_name = f"Part_{split_num}"
        split_dir = get_split_dir(split_num)

        if not os.path.exists(split_dir):
            print("Không tìm thấy:", split_dir)
            continue

        for view in VIEWS:
            view_dir = os.path.join(split_dir, view)
            json_path = os.path.join(split_dir, f"{view}.json")

            if not os.path.exists(view_dir):
                print("Không tìm thấy folder:", view_dir)
                continue

            if not os.path.exists(json_path):
                print("Không tìm thấy json:", json_path)
                continue

            data = load_json(json_path)
            items = extract_rows_from_json(data)

            for item in items:
                video_id = item["video_id"]
                gloss = item["gloss"]

                if video_id is None:
                    continue

                video_file = normalize_video_name(video_id)
                video_path = os.path.join(view_dir, video_file)

                all_rows.append({
                    "split_type": split_type,
                    "part": part_name,
                    "split": split_name,
                    "view": view,
                    "video_id": video_file.replace(".mp4", ""),
                    "gloss": gloss,
                    "video_path": video_path,
                    "exists": os.path.exists(video_path)
                })

    return pd.DataFrame(all_rows)


train_df = build_metadata(TRAIN_SPLITS, "train")
test_df = build_metadata(TEST_SPLITS, "test")

train_df = train_df[train_df["exists"] == True].copy()
test_df = test_df[test_df["exists"] == True].copy()

train_df.to_csv("/kaggle/working/train.csv", index=False)
test_df.to_csv("/kaggle/working/test.csv", index=False)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

print("\nTrain sample:")
display(train_df.head())

print("\nTest sample:")
display(test_df.head())

print("\nTrain theo part:")
print(train_df["part"].value_counts())

print("\nTest theo part:")
print(test_df["part"].value_counts())

print("\nTrain theo view:")
print(train_df["view"].value_counts())

print("\nTest theo view:")
print(test_df["view"].value_counts())

print("\nSố gloss train:", train_df["gloss"].nunique())
print("Số gloss test:", test_df["gloss"].nunique())

print("\nĐã lưu:")
print("/kaggle/working/train.csv")
print("/kaggle/working/test.csv")

Train: (21217, 8)
Test: (3536, 8)

Train sample:


,split_type,part,split,view,video_id,gloss,video_path,exists
0,train,Part_1,split_1,front_view,000000,Anh,/kaggle/input/datasets/bowboochua9/vsl400/Part...,True
1,train,Part_1,split_1,front_view,000001,Anh,/kaggle/input/datasets/bowboochua9/vsl400/Part...,True
2,train,Part_1,split_1,front_view,000002,Chị,/kaggle/input/datasets/bowboochua9/vsl400/Part...,True
3,train,Part_1,split_1,front_view,000003,Chị,/kaggle/input/datasets/bowboochua9/vsl400/Part...,True
4,train,Part_1,split_1,front_view,000004,Em,/kaggle/input/datasets/bowboochua9/vsl400/Part...,True



Test sample:


,split_type,part,split,view,video_id,gloss,video_path,exists
0,test,Part_7,split_7,front_view,021239,Màu xanh da trời,/kaggle/input/datasets/bowboochua9/vsl400/Part...,True
1,test,Part_7,split_7,front_view,021240,Màu hồng,/kaggle/input/datasets/bowboochua9/vsl400/Part...,True
2,test,Part_7,split_7,front_view,021241,Màu tím,/kaggle/input/datasets/bowboochua9/vsl400/Part...,True
3,test,Part_7,split_7,front_view,021242,Màu nâu,/kaggle/input/datasets/bowboochua9/vsl400/Part...,True
4,test,Part_7,split_7,front_view,021243,Quả dâu,/kaggle/input/datasets/bowboochua9/vsl400/Part...,True



Train theo part:
part
Part_1    3537
Part_2    3536
Part_3    3536
Part_4    3536
Part_5    3536
Part_6    3536
Name: count, dtype: int64

Test theo part:
part
Part_7    3536
Name: count, dtype: int64

Train theo view:
view
front_view    21217
Name: count, dtype: int64

Test theo view:
view
front_view    3536
Name: count, dtype: int64

Số gloss train: 400
Số gloss test: 400

Đã lưu:
/kaggle/working/train.csv
/kaggle/working/test.csv


In [8]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GLOG_minloglevel"] = "3"

warnings.filterwarnings("ignore")

import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)
absl.logging.set_stderrthreshold("error")

lay tat ca npy them vao working

In [9]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

MAX_FRAMES = 60
FEATURE_DIM = 258

POSE_MODEL = "/kaggle/working/mp_models/pose_landmarker.task"
HAND_MODEL = "/kaggle/working/mp_models/hand_landmarker.task"

OUT_ROOT = "/kaggle/working/keypoints"
TRAIN_NPY_DIR = f"{OUT_ROOT}/train"
TEST_NPY_DIR = f"{OUT_ROOT}/test"

os.makedirs(TRAIN_NPY_DIR, exist_ok=True)
os.makedirs(TEST_NPY_DIR, exist_ok=True)


BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode

PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions

HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions


def create_landmarkers():
    pose_options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=POSE_MODEL),
        running_mode=VisionRunningMode.VIDEO,
        num_poses=1
    )

    hand_options = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=HAND_MODEL),
        running_mode=VisionRunningMode.VIDEO,
        num_hands=2
    )

    pose_landmarker = PoseLandmarker.create_from_options(pose_options)
    hand_landmarker = HandLandmarker.create_from_options(hand_options)

    return pose_landmarker, hand_landmarker


def extract_pose(pose_result):
    if pose_result.pose_landmarks and len(pose_result.pose_landmarks) > 0:
        landmarks = pose_result.pose_landmarks[0]
        pose = np.array(
            [[lm.x, lm.y, lm.z, lm.visibility] for lm in landmarks],
            dtype=np.float32
        ).flatten()
    else:
        pose = np.zeros(33 * 4, dtype=np.float32)

    return pose


def extract_hands(hand_result):
    left = np.zeros(21 * 3, dtype=np.float32)
    right = np.zeros(21 * 3, dtype=np.float32)

    if not hand_result.hand_landmarks:
        return left, right

    for i, landmarks in enumerate(hand_result.hand_landmarks):
        hand_vec = np.array(
            [[lm.x, lm.y, lm.z] for lm in landmarks],
            dtype=np.float32
        ).flatten()

        handedness = hand_result.handedness[i][0].category_name

        if handedness == "Left":
            left = hand_vec
        elif handedness == "Right":
            right = hand_vec

    return left, right


def extract_keypoints_tasks(pose_result, hand_result):
    pose = extract_pose(pose_result)
    lh, rh = extract_hands(hand_result)

    return np.concatenate([pose, lh, rh]).astype(np.float32)


def pad_or_cut(sequence, max_frames=60):
    sequence = np.array(sequence, dtype=np.float32)

    if len(sequence) == 0:
        return np.zeros((max_frames, FEATURE_DIM), dtype=np.float32)

    if len(sequence) > max_frames:
        idx = np.linspace(0, len(sequence) - 1, max_frames).astype(int)
        sequence = sequence[idx]

    elif len(sequence) < max_frames:
        pad_len = max_frames - len(sequence)
        pad = np.zeros((pad_len, FEATURE_DIM), dtype=np.float32)
        sequence = np.vstack([sequence, pad])

    return sequence.astype(np.float32)


def video_to_npy(video_path, save_path, max_frames=60):
    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 0:
        fps = 30

    sequence = []

    pose_landmarker, hand_landmarker = create_landmarkers()

    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        timestamp_ms = int(frame_idx * 1000 / fps)

        pose_result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)
        hand_result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)

        keypoints = extract_keypoints_tasks(pose_result, hand_result)
        sequence.append(keypoints)

        frame_idx += 1

    cap.release()
    pose_landmarker.close()
    hand_landmarker.close()

    sequence = pad_or_cut(sequence, max_frames=max_frames)
    np.save(save_path, sequence)

    return sequence.shape

E0000 00:00:1781261947.512153      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781261947.568443      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781261948.003099      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781261948.003143      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781261948.003146      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781261948.003148      22 computation_placer.cc:177] computation placer already registered. Please check linka

In [10]:
import os
import sys

class SuppressOutput:
    def __enter__(self):
        self.stdout_fd = sys.stdout.fileno()
        self.stderr_fd = sys.stderr.fileno()

        self.saved_stdout_fd = os.dup(self.stdout_fd)
        self.saved_stderr_fd = os.dup(self.stderr_fd)

        self.devnull_fd = os.open(os.devnull, os.O_WRONLY)

        os.dup2(self.devnull_fd, self.stdout_fd)
        os.dup2(self.devnull_fd, self.stderr_fd)

    def __exit__(self, exc_type, exc_val, exc_tb):
        os.dup2(self.saved_stdout_fd, self.stdout_fd)
        os.dup2(self.saved_stderr_fd, self.stderr_fd)

        os.close(self.devnull_fd)
        os.close(self.saved_stdout_fd)
        os.close(self.saved_stderr_fd)

In [11]:
def convert_df_to_npy(df, out_dir, save_csv_path):
    new_rows = []
    os.makedirs(out_dir, exist_ok=True)

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        video_path = row["video_path"]

        safe_video_id = str(row["video_id"]).replace("/", "_").replace("\\", "_")

        # Tên file không bị trùng giữa Part/split/view
        npy_name = f'{row["part"]}_{row["split"]}_{row["view"]}_{safe_video_id}.npy'
        npy_path = os.path.join(out_dir, npy_name)

        new_row = row.to_dict()
        new_row["npy_name"] = npy_name
        new_row["npy_path"] = npy_path

        try:
            # Nếu chạy lại trong cùng session thì bỏ qua file đã có
            if os.path.exists(npy_path):
                shape = np.load(npy_path).shape
            else:
                with SuppressOutput():
                    shape = video_to_npy(video_path, npy_path, max_frames=MAX_FRAMES)

            new_row["npy_exists"] = os.path.exists(npy_path)
            new_row["npy_shape"] = str(shape)

        except Exception as e:
            print("Lỗi video:", video_path)
            print("Lỗi:", e)

            new_row["npy_exists"] = False
            new_row["npy_shape"] = None

        new_rows.append(new_row)

        # Cứ 100 video lưu tạm progress
        if len(new_rows) % 100 == 0:
            temp_df = pd.DataFrame(new_rows)
            temp_df.to_csv(save_csv_path.replace(".csv", "_progress.csv"), index=False)

    result_df = pd.DataFrame(new_rows)
    result_df.to_csv(save_csv_path, index=False)
    return result_df

In [12]:
half = len(train_df) // 2

train_df_50 = train_df.iloc[-3000:].reset_index(drop=True).copy()

train_npy_df = convert_df_to_npy(
    train_df_50,
    TRAIN_NPY_DIR,
    "/kaggle/working/train.csv"
)

  0%|          | 0/3000 [00:00<?, ?it/s]INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1781261960.562142      94 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781261960.596843      96 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781261960.638048     100 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781261960.662508     102 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781261960.752210      94 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSI

In [13]:
import os

npy_count = 0
for root, dirs, files in os.walk("/kaggle/working/keypoints"):
    for f in files:
        if f.endswith(".npy"):
            npy_count += 1

print("Số npy hiện có:", npy_count)

Số npy hiện có: 3000


In [14]:
!zip -rq /kaggle/working/vsl400_checkpoint_back_3000.zip \
/kaggle/working/keypoints \
/kaggle/working/*.csv

In [15]:
import os

for f in os.listdir("/kaggle/working"):
    if f.endswith(".zip"):
        path = "/kaggle/working/" + f
        
        print(f, round(os.path.getsize(path) / 1024 / 1024, 2), "MB")

vsl400_checkpoint_back_3000.zip 139.63 MB


In [16]:
from IPython.display import FileLink, display

display(FileLink("vsl400_checkpoint_back_3000.zip"))

/kaggle/working/vsl400_checkpoint_back_3000.zip

In [17]:
!rm -rf /kaggle/working/keypoints
!mkdir -p /kaggle/working/keypoints/train
!mkdir -p /kaggle/working/keypoints/test